In [1]:
# Import libraries for data handling and association rules
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

ModuleNotFoundError: No module named 'mlxtend'

In [2]:
%pip install mlxtend

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.4 MB 1.2 MB/s eta 0:00:01
   ----------------------- ---------------- 0.8/1.4 MB 1.0 MB/s eta 0:00:01
   ------------------------------ --------- 1.0/1.4 MB 1.1 MB/s eta 0:00:01
   ------------------------------ --------- 1.0/1.4 MB 1.1 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 995.0 kB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.


In [10]:
# Import libraries for data handling and association rule mining
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

In [11]:
# Load the Groceries dataset from the local CSV file
df = pd.read_csv("Groceries_dataset.csv")

print(df.head())
print("Dataset shape:", df.shape)

   Member_number        Date   itemDescription
0           1808  21-07-2015    tropical fruit
1           2552  05-01-2015        whole milk
2           2300  19-09-2015         pip fruit
3           1187  12-12-2015  other vegetables
4           3037  01-02-2015        whole milk
Dataset shape: (38765, 3)


In [12]:
# Remove missing values and duplicate records
df = df.dropna()
df = df.drop_duplicates()

df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

print(df.isnull().sum())

Member_number      0
Date               0
itemDescription    0
dtype: int64


In [13]:
# Group purchased products by customer number and date
baskets = df.groupby(["Member_number", "Date"])["itemDescription"].apply(list)

print(baskets.head())
print("Total baskets:", len(baskets))

Member_number  Date      
1000           2014-06-24                    [whole milk, pastry, salty snack]
               2015-03-15    [sausage, whole milk, semi-finished bread, yog...
               2015-05-27                           [soda, pickled vegetables]
               2015-07-24                       [canned beer, misc. beverages]
               2015-11-25                          [sausage, hygiene articles]
Name: itemDescription, dtype: object
Total baskets: 14963


In [14]:
# Convert each basket into unique products and create a binary matrix
transactions = baskets.apply(lambda items: list(set(items)))

all_items = sorted(set(item for basket in transactions for item in basket))

basket_matrix = pd.DataFrame(
    [{item: (item in basket) for item in all_items} for basket in transactions],
    index=transactions.index
)

print(basket_matrix.iloc[:5, :8])

                          Instant food products  UHT-milk  abrasive cleaner  \
Member_number Date                                                            
1000          2014-06-24                  False     False             False   
              2015-03-15                  False     False             False   
              2015-05-27                  False     False             False   
              2015-07-24                  False     False             False   
              2015-11-25                  False     False             False   

                          artif. sweetener  baby cosmetics   bags  \
Member_number Date                                                  
1000          2014-06-24             False           False  False   
              2015-03-15             False           False  False   
              2015-05-27             False           False  False   
              2015-07-24             False           False  False   
              2015-11-25        

In [18]:
# Generate frequent itemsets with a lower minimum support
frequent_itemsets = apriori(
    basket_matrix,
    min_support=0.005,
    use_colnames=True
)

print(frequent_itemsets.head(10))

    support                    itemsets
0  0.021386       frozenset({UHT-milk})
1  0.008087  frozenset({baking powder})
2  0.033950           frozenset({beef})
3  0.021787        frozenset({berries})
4  0.016574      frozenset({beverages})
5  0.045312   frozenset({bottled beer})
6  0.060683  frozenset({bottled water})
7  0.037626    frozenset({brown bread})
8  0.035220         frozenset({butter})
9  0.017577    frozenset({butter milk})


In [21]:
# Generate association rules with a lower confidence threshold
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.10
)

rules = rules.sort_values(by="lift", ascending=False)

print(rules[["antecedents", "consequents",
             "support", "confidence", "lift"]].head(10))

                   antecedents                    consequents   support  \
5     frozenset({frankfurter})  frozenset({other vegetables})  0.005146   
0    frozenset({bottled beer})        frozenset({whole milk})  0.007151   
14        frozenset({sausage})        frozenset({whole milk})  0.008955   
7      frozenset({newspapers})        frozenset({whole milk})  0.005614   
4   frozenset({domestic eggs})        frozenset({whole milk})  0.005280   
6     frozenset({frankfurter})        frozenset({whole milk})  0.005280   
11           frozenset({pork})        frozenset({whole milk})  0.005012   
10      frozenset({pip fruit})        frozenset({whole milk})  0.006616   
3    frozenset({citrus fruit})        frozenset({whole milk})  0.007151   
15  frozenset({shopping bags})        frozenset({whole milk})  0.006349   

    confidence      lift  
5     0.136283  1.116150  
0     0.157817  0.999330  
14    0.148394  0.939663  
7     0.144330  0.913926  
4     0.142342  0.901341  
6     0.1398

In [22]:
# Keep rules with lift greater than 1 and display the strongest associations
strong_rules = rules[rules["lift"] > 1]

print(strong_rules[["antecedents", "consequents",
                    "support", "confidence", "lift"]].head(10))

                antecedents                    consequents   support  \
5  frozenset({frankfurter})  frozenset({other vegetables})  0.005146   

   confidence     lift  
5    0.136283  1.11615  


In [23]:
# Check the number of generated rules and their lift values
print("Total rules:", len(rules))
print("Maximum lift:", rules["lift"].max())

Total rules: 19
Maximum lift: 1.1161495948191098


In [24]:
# Keep rules with lift greater than 1 and display the strongest associations
strong_rules = rules[rules["lift"] > 1]

print(strong_rules[["antecedents", "consequents",
                    "support", "confidence", "lift"]].head(10))

                antecedents                    consequents   support  \
5  frozenset({frankfurter})  frozenset({other vegetables})  0.005146   

   confidence     lift  
5    0.136283  1.11615  
